# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print metadata info
meta_json = dataset.metadata.to_json()
print(f"{meta_json['name']}: {meta_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id and fields
print("Record Sets Overview:")
for rs in dataset.record_sets:
    print(f"- Record set name: {rs.name}, @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields and their @ids (columns):")
        for field in rs.fields:
            print(f"    - {field.name} (field @id: {field.id}, column @id: {getattr(field, 'column', None)})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
# Load data for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs_id}, data shape: {df.shape}")

# Select the main record set for this analysis. Consult overview for suitable one.
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
import numpy as np

# Choose a numeric field for demonstration
# For mental health survey, likely candidates: GAD-7, PHQ-9, PSQ total scores.
# Select by inspecting columns.
numeric_fields = [c for c in dataframes[main_record_set_id].columns if any(s in c.lower() for s in ['gad', 'phq', 'psq', 'score', 'total'])]
print('Available numeric fields for analysis:', numeric_fields)

if numeric_fields:
    numeric_field = numeric_fields[0] # Use the first as example
    threshold = dataframes[main_record_set_id][numeric_field].mean() # Example: filter above mean
    
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df[[numeric_field]].head())

    # Normalize
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records (z-score normalization):")
    display(filtered_df[[numeric_field, col_norm]].head())

    # Try grouping by a demographic, e.g., 'gender', if exists
    possible_group_fields = [c for c in filtered_df.columns if 'gender' in c.lower() or 'age' in c.lower() or 'education' in c.lower()]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found in this record set.")
else:
    print("No numeric field found for this EDA demo.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group_field exists
    if 'group_field' in locals():
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Kilifi County Mental Health Survey dataset using the Croissant schema and `mlcroissant`.
- The dataset includes several record sets and fields, including mental health assessment scores (e.g., GAD-7, PHQ-9, PSQ) and demographic data.
- We demonstrated basic data exploration, normalization, and simple visualizations. For advanced analysis, you may link more fields by their `@id` as found in the dataset schema.